# 因子构造 
**AED**因子构建    
取agent的行动 $action_{i,s,t}$  
其中，$i$ 为agent的编号，$s$ 为agent的行动，$t$ 为时间窗口     
对于每一个证券-时间窗口，求所有agent的行动的平均值与标准差的比值，作为AED因子。     

如果所有agent都做出了高的决策，则AED大，如果所有agent都做出了低的决策，则AED小，如果agent的决策差异大，则AED也小。   

$$
AED_{s,t} = \frac{\sum_{i=1}^{n} action_{i,s,t}}{n} / \sqrt{\frac{\sum_{i=1}^{n} (action_{i,s,t} - \bar{action_{s,t}})^2}{n}}
$$

按照语义，由于训练是 拆分证券 + 多次训练 + 多次保存，一般来说，一次任务中会包括多个目录，多个jsonl。一般来说，一个task_prefix代表着同一个实验，其数据共用。   

## 导入库

In [6]:
import json  
import os
import re
import polars as pl
from pathlib import Path
import warnings

## 超参数

In [7]:
TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

## 读取数据
数据的保存路径为：    

```
DATA_BASE_DIR  
|- TASK_ID1
|   |- Node1
|   |   |- performance_and_record_0.json   
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- Node2
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- ...
|- TASK_ID2
|   |- Node1
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
```  

其中，task_id的构成为 时间_中缀_uuid，例如：20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57  
指定中缀，选取所有中缀一样的task_id，得到所有node的路径  

In [8]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

node_paths


['/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014488_6824',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772031879_25181',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772031891_9646',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772031901_11800',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014501_6831',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772033480_1918',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014595_27084',
 '/home/frank/files/programs/Graduatio

In [9]:
# 获取其下所有performance_and_record_*.jsonl文件的路径
jsonl_paths = []
for node_path in node_paths:
    all_files = os.listdir(node_path)
    perf_and_rewa_jsonl_files = [file for file in all_files if file.endswith('.jsonl') and 'performance_and_reward_' in file]
    jsonl_paths.extend(
        os.path.join(node_path, file)
        for file in perf_and_rewa_jsonl_files
    )
jsonl_paths

['/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014488_6824/performance_and_reward_10.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014488_6824/performance_and_reward_8.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014488_6824/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014488_6824/performance_and_reward_9.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014488_6824/performance_and_reward_5.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7/node_1772014488_6824/performance_and_reward_3

先用scan获取所有的jsonl  
使用polars的concat方法，合并所有json，字段由performance_and_record约定好  
展示测试数据  

In [10]:
lazy_frames = [pl.scan_ndjson(f) for f in jsonl_paths]
lf = pl.concat(lazy_frames)   # 得到 LazyFrame
lf.head().collect(engine = 'auto')


year,month,portfolio,data
i64,i64,list[str],struct[4]
2013,1,"[""600271""]","{[0.5, 0.5],[0.0074, -0.06, … 1.0],[0.1795, 0.1633, … 1.4157],0.0014}"
2013,1,"[""000400""]","{[0.6325, 0.3675],[-0.0381, -0.0948, … 0.9487],[-0.4265, -0.8336, … 1.3001],-0.0532}"
2013,1,"[""600486""]","{[0.7508, 0.2492],[0.0068, -0.0883, … 0.8099],[0.171, -0.6474, … 0.9872],-0.0063}"
2013,1,"[""600169""]","{[0.4002, 0.5998],[-0.0194, -0.1256, … 0.9711],[-0.1777, -1.72, … 1.3505],-0.0459}"
2013,1,"[""000739""]","{[1.0, 0.0],[0.3703, -0.1092, … -0.0],[5.0052, -1.2482, … -0.8391],0.3503}"


## 处理数据  
加载的lf，其结构如上所示，需要进行解析   

>- 1.基线回归为单证券，因此，需要验证portfolio中列表长度，如果有长度大于1的列表，需要警告，并且取[0]  
>- 2. data比较复杂，由多项fields组成，包括decision_weights, performance, normalized_performance, reward； 其中，如果是多证券，decision_weights, 是list结构，需要判断是否有长度大于1的列表，如果有，需要警告，并且取[0]; performance 和 normalized_performance是固定len=5的list，对应字段return, -vol, sharp, -maxdrawdwon, devisification，需要展开为对应字段   

In [11]:
# 展示fields
df_demo = lf.head().collect()
df_demo.with_columns(pl.col('data').struct.unnest()).select(pl.all().exclude('data'))

year,month,portfolio,decision_weights,performance,normalized_performance,reward
i64,i64,list[str],list[f64],list[f64],list[f64],f64
2013,1,"[""600271""]","[0.5, 0.5]","[0.0074, -0.06, … 1.0]","[0.1795, 0.1633, … 1.4157]",0.0014
2013,1,"[""000400""]","[0.6325, 0.3675]","[-0.0381, -0.0948, … 0.9487]","[-0.4265, -0.8336, … 1.3001]",-0.0532
2013,1,"[""600486""]","[0.7508, 0.2492]","[0.0068, -0.0883, … 0.8099]","[0.171, -0.6474, … 0.9872]",-0.0063
2013,1,"[""600169""]","[0.4002, 0.5998]","[-0.0194, -0.1256, … 0.9711]","[-0.1777, -1.72, … 1.3505]",-0.0459
2013,1,"[""000739""]","[1.0, 0.0]","[0.3703, -0.1092, … -0.0]","[5.0052, -1.2482, … -0.8391]",0.3503


In [12]:
# 定义字段
perf_fields = ["return", "neg_vol", "sharp", "neg_maxdrawdown", "diversification"]

# 1. 在 LazyFrame 上完成 unnest，再 collect（单次扫描，避免先 collect 再在 Python 里 unnest）
lf_unnested = lf.with_columns(pl.col("data").struct.unnest()).drop("data")
df_flat = lf_unnested.collect()  # 需要子集时可改为 lf_unnested.head(n).collect()

# 2. portfolio：检查是否有多证券，有则告警并只保留 [0]
if (df_flat["portfolio"].list.len() > 1).any():
    warnings.warn("发现 portfolio 列表长度 > 1（多证券），已取 [0] 作为单证券基线。")
df_flat = df_flat.with_columns(pl.col("portfolio").list.get(0).alias("portfolio"))

# 3. decision_weights：多证券时告警并取 [0]
if (df_flat["decision_weights"].list.len() > 1).any():
    warnings.warn("发现 decision_weights 列表长度 > 1，已取 [0]。")
df_flat = df_flat.with_columns(pl.col("decision_weights").list.get(0).alias("decision_weights"))

# 4. performance / normalized_performance：固定 len=5，展开为多列（return, -vol, sharp, -maxdrawdown, diversification）
df_flat = df_flat.with_columns(
    pl.col("performance").list.to_struct(fields=[f"{f}" for f in perf_fields]).struct.unnest()
).drop("performance")
df_flat = df_flat.with_columns(
    pl.col("normalized_performance").list.to_struct(fields=[f"normalized_{f}" for f in perf_fields]).struct.unnest()
).drop("normalized_performance")
# reward 已是标量，无需处理

/tmp/ipykernel_94608/3039891457.py:15: UserWarning: 发现 decision_weights 列表长度 > 1，已取 [0]。
  warnings.warn("发现 decision_weights 列表长度 > 1，已取 [0]。")


In [13]:
df_flat.head()

year,month,portfolio,decision_weights,reward,return,neg_vol,sharp,neg_maxdrawdown,diversification,normalized_return,normalized_neg_vol,normalized_sharp,normalized_neg_maxdrawdown,normalized_diversification
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2013,1,"""600271""",0.5,0.0014,0.0074,-0.06,-0.1881,-0.2131,1.0,0.1795,0.1633,-0.2454,0.0245,1.4157
2013,1,"""000400""",0.6325,-0.0532,-0.0381,-0.0948,0.3392,-0.1288,0.9487,-0.4265,-0.8336,1.5705,0.6676,1.3001
2013,1,"""600486""",0.7508,-0.0063,0.0068,-0.0883,0.2059,-0.1941,0.8099,0.171,-0.6474,1.1112,0.17,0.9872
2013,1,"""600169""",0.4002,-0.0459,-0.0194,-0.1256,-0.2604,-0.4933,0.9711,-0.1777,-1.72,-0.4944,-2.1124,1.3505
2013,1,"""000739""",1.0,0.3503,0.3703,-0.1092,0.4418,-0.1071,-0.0,5.0052,-1.2482,1.9237,0.833,-0.8391


In [14]:
# 转为lf，方便后续处理
lf_flat = df_flat.lazy()

## 构建因子
为了减少处理难度，先筛选需要的列  

- year   
- month    
- portfolio    
- decision_weights     
- return (表示year-month下一个月的收益)   

In [15]:
needed_cols = ['year','month','portfolio','decision_weights','return']
filtered_lf = lf_flat.select(
    pl.col(needed_cols)
)
filtered_lf.head().collect()

year,month,portfolio,decision_weights,return
i64,i64,str,f64,f64
2013,1,"""600271""",0.5,0.0074
2013,1,"""000400""",0.6325,-0.0381
2013,1,"""600486""",0.7508,0.0068
2013,1,"""600169""",0.4002,-0.0194
2013,1,"""000739""",1.0,0.3703


按照AED因子的定义，构建因子，按照year,month,portfolio进行分组，求每一个组decision_weights的 平均值 / 标准差，作为AED因子  

In [16]:
aed_lf = filtered_lf.group_by(['year','month','portfolio']).agg(
    (pl.col('decision_weights').mean() / pl.col('decision_weights').std()).alias('AED'),
    pl.col('return').first().alias('return')
)
aed_lf.head().collect()

year,month,portfolio,AED,return
i64,i64,str,f64,f64
2016,10,"""002461""",1.6237,-0.0214
2010,6,"""002238""",3.113288,0.061
2013,10,"""002245""",3.120258,0.0702
2015,10,"""002205""",3.206781,0.2277
2017,1,"""002698""",3.564664,0.0365


最后，将year,month合并为date  

In [17]:
aed_lf = aed_lf.with_columns(
    pl.date(pl.col('year'), pl.col('month'), 1).alias('date')
).drop(['year','month']).select(['date','portfolio','AED','return'])
aed_lf.head().collect()

date,portfolio,AED,return
date,str,f64,f64
2013-09-01,"""600222""",2.373044,0.0397
2021-01-01,"""002577""",2.252471,-0.0115
2021-08-01,"""002612""",2.48644,-0.1237
2020-10-01,"""002045""",4.329643,-0.0609
2018-10-01,"""300232""",2.013763,0.0408


## 保存数据  
保存数据为parquet文件  

In [18]:
if SAVE:
    dir_path = Path(SAVE_BASELINE_REG_DIR)
    dir_path.mkdir(parents=True, exist_ok=True)
    aed_lf.collect().write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, f'基准回归-AED因子.parquet'))
    print(f'保存成功，路径为:{os.path.join(SAVE_BASELINE_REG_DIR, f"基准回归-AED因子.parquet")}')
else:
    print('请设置SAVE=True，以保存数据')

保存成功，路径为:/home/frank/files/programs/GraduationThesis/empirical/baseline1/baseline_reg/基准回归-AED因子.parquet
